# model-v2
Generated from `code/business_entity_resolution/src/run_two_stage.py` by `jobs/build_notebook.py`. Edit the script, not this notebook.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 scipy==1.18.1 scikit-learn==1.9.1 sparse-dot-topn==1.2.0 lightgbm==4.7.0

In [ ]:
import os
os.makedirs('/tmp/src', exist_ok=True)
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, dirs, files)

In [ ]:
%%writefile /tmp/src/model.py
"""Match model: LightGBM binary classifier on candidate pairs (features feat-v2, table 6).

Feature sets (the France question: French test candidates are found by fewer search methods, median 2 vs 4):
  A  all 86 features;
  B  A without f_n_methods (how many search methods found the pair);
  C  B without the pruner-derived features (f_p_u50, f_p_gap, f_list_rank): the blocking pruner itself uses the
     method count and the per-method ranks, so its score can carry the same bias.
Per-method scores and ranks stay in every set (they are similarities; "not found" is NaN).

Training S1 exclude the 100k train-split S1 the blocking pruner was trained on (their pruner scores and lists
are in-sample); `A_all` keeps them, to measure that effect. Early stopping uses a held-out tenth of the training
S1 (by S1, never by pair); validation S1 are never used for training or early stopping.
"""
from __future__ import annotations

import lightgbm as lgb
import numpy as np
import polars as pl

SEED = 42
PRUNER_DERIVED = ("f_p_u50", "f_p_gap", "f_list_rank")
PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=127, min_data_in_leaf=100, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, seed=SEED, deterministic=True, verbose=-1,
              num_threads=0)
ROUNDS, EARLY_STOP = 3000, 100


def feature_sets(cols: list[str]) -> dict[str, list[str]]:
    f = [c for c in cols if c.startswith("f_")]
    b = [c for c in f if c != "f_n_methods"]
    return {"A": f, "B": b, "C": [c for c in b if c not in PRUNER_DERIVED]}


def early_stop_mask(s1: pl.Series, share: float = 0.1) -> np.ndarray:
    """True for rows whose S1 falls in the held-out share (stable hash of the id, same S1 -> same side)."""
    return ((s1.hash(seed=SEED) % 1000) < int(share * 1000)).to_numpy()


def train(x: pl.DataFrame, y: np.ndarray, feats: list[str], log=print) -> lgb.Booster:
    """x: s1 + feature columns; y: 0/1. Early stopping on a held-out tenth of the S1."""
    hold = early_stop_mask(x["s1"])
    m = x.select(feats).to_numpy()
    dtr = lgb.Dataset(m[~hold], y[~hold], feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(m[hold], y[hold], reference=dtr)
    booster = lgb.train(PARAMS, dtr, ROUNDS, valid_sets=[dva], valid_names=["early_stop"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
    log(f"    {len(feats)} features, {int((~hold).sum()):,} fit rows, {int(hold.sum()):,} early-stop rows, "
        f"best iteration {booster.best_iteration}, early-stop logloss {booster.best_score['early_stop']['binary_logloss']:.4f}")
    return booster


def predict(booster: lgb.Booster, x: pl.DataFrame, chunk: int = 2_000_000) -> np.ndarray:
    feats = booster.feature_name()
    return np.concatenate([booster.predict(x[lo:lo + chunk].select(feats).to_numpy(), num_iteration=booster.best_iteration)
                           for lo in range(0, x.height, chunk)] or [np.empty(0)])


def importance(booster: lgb.Booster, name: str) -> pl.DataFrame:
    g = booster.feature_importance("gain")
    return (pl.DataFrame({"variant": name, "feature": booster.feature_name(), "gain": g.astype(float)})
            .with_columns((pl.col("gain") / pl.col("gain").sum()).alias("gain_share")).sort("gain", descending=True))


In [ ]:
%%writefile /tmp/src/selection.py
"""Match selection: from scored candidate pairs to the predicted matches per S1, tuned for macro F0.5.

Rules (all tuned on validation, never on test):
- threshold: keep a pair only if its match probability p >= tau;
- one owner: every S2/S3 record matches at most one S1 (EDA: 0 of 7.6M train records match two), so each
  candidate is kept only for the S1 that gives it the highest p;
- margin: keep a pair only if p >= margin * (best p of that S1), which drops weak extra matches next to a strong one.
F0.5 weighs precision twice, so a wrong match costs more than a missed one: the tuned threshold is usually high.

f05_table() is a vectorised version of metric.py (same definition, checked against it in the tests and in
run_model.py): macro F0.5 over ALL evaluated S1, an S1 without predictions scores 1.0 if it has no true match.
"""
from __future__ import annotations

import itertools

import polars as pl

TAUS = [round(0.05 * i, 2) for i in range(2, 20)]  # 0.10 .. 0.95
MARGINS = [0.0, 0.3, 0.5, 0.7]
BETA2 = 0.25


def select(scored: pl.DataFrame, tau: float, margin: float = 0.0, one_owner: bool = True, tau2: float | None = None,
           sim: float = 0.9, owner_delta: float = 0.0) -> pl.DataFrame:
    """scored: s1, cand, p (+ s_sim_name, s_num_eq for the secondary rule). Returns the kept (s1, cand, p) rows.
    tau2: also keep a pair with tau2 <= p < tau if it strongly resembles a confident candidate of the same S1
    (s_sim_name >= sim and the same first house number). owner_delta: with one_owner, an S2/S3 claimed by two S1
    goes to the higher p only if it leads by more than owner_delta; otherwise it is dropped from both."""
    keep = pl.col("p") >= tau
    if tau2 is not None and tau2 < tau:
        keep = keep | ((pl.col("p") >= tau2) & (pl.col("s_sim_name") >= sim) & (pl.col("s_num_eq") >= 1)).fill_null(False)
    x = scored.filter(keep)
    if one_owner:  # ties: the smaller s1 id wins (deterministic)
        x = x.sort(["cand", "p", "s1"], descending=[False, True, False])
        if owner_delta > 0:
            lead = pl.col("p") - pl.col("p").shift(-1).over("cand")
            x = x.with_columns(pl.int_range(pl.len()).over("cand").alias("_r"), lead.fill_null(1.0).alias("_lead"))
            x = x.filter((pl.col("_r") == 0) & (pl.col("_lead") > owner_delta)).drop("_r", "_lead")
        else:
            x = x.unique("cand", keep="first", maintain_order=True)
    if margin > 0:
        x = x.filter(pl.col("p") >= margin * pl.col("p").max().over("s1"))
    return x.select("s1", "cand", "p")


def f05_table(pred: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame) -> pl.DataFrame:
    """Per evaluated S1: n_pred, n_true, tp, precision, recall, f05. pred / truth: s1, cand; s1s: s1 (+ any
    attribute columns, e.g. country), the S1 to evaluate. S1 missing from pred count as empty predictions."""
    keys = s1s.select("s1")
    pred = pred.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    truth = truth.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    tp = pred.join(truth, on=["s1", "cand"], how="semi").group_by("s1").len("tp")
    x = (s1s.join(pred.group_by("s1").len("n_pred"), on="s1", how="left")
         .join(truth.group_by("s1").len("n_true"), on="s1", how="left")
         .join(tp, on="s1", how="left")
         .with_columns(pl.col("n_pred", "n_true", "tp").fill_null(0)))
    prec = pl.when(pl.col("n_pred") > 0).then(pl.col("tp") / pl.col("n_pred"))
    rec = pl.when(pl.col("n_true") > 0).then(pl.col("tp") / pl.col("n_true"))
    f = (pl.when(pl.col("n_true") == 0).then((pl.col("n_pred") == 0).cast(pl.Float64))
         .when(pl.col("tp") == 0).then(0.0)
         .otherwise((1 + BETA2) * prec * rec / (BETA2 * prec + rec)))
    return x.with_columns(prec.alias("precision"), rec.alias("recall"), f.alias("f05"))


def summary(tab: pl.DataFrame, by: str | None = None) -> pl.DataFrame:
    """Macro F0.5, mean precision / recall (over S1 where defined), per group and overall (+ singletons)."""
    aggs = [pl.len().alias("s1"), pl.col("f05").mean().alias("macro_f05"), pl.col("precision").mean().alias("precision"),
            pl.col("recall").mean().alias("recall"), pl.col("n_pred").mean().alias("pred_per_s1")]
    kind = pl.when(pl.col("n_true") == 0).then(pl.lit("singleton")).otherwise(pl.lit("non_singleton"))
    parts = [tab.select(pl.lit("overall").alias("group"), *aggs),
             tab.with_columns(kind.alias("_k")).group_by("_k").agg(aggs).sort("_k")
             .select(pl.format("kind={}", "_k").alias("group"), *[a.meta.output_name() for a in aggs])]
    if by:
        parts.insert(1, tab.group_by(by).agg(aggs).sort(by).select(pl.format(f"{by}={{}}", by).alias("group"),
                                                                     *[a.meta.output_name() for a in aggs]))
    return pl.concat(parts, how="vertical_relaxed")


def tune(scored: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame, taus=TAUS, margins=MARGINS,
         owners=(True, False)) -> pl.DataFrame:
    """Macro F0.5 on s1s for every (tau, margin, one_owner). Sorted best first."""
    rows = []
    for tau, margin, owner in itertools.product(taus, margins, owners):
        tab = f05_table(select(scored, tau, margin, owner), truth, s1s)
        rows.append({"tau": tau, "margin": margin, "one_owner": owner, "macro_f05": tab["f05"].mean(),
                     "pred_per_s1": tab["n_pred"].mean()})
    return pl.DataFrame(rows).sort("macro_f05", descending=True)


In [ ]:
%%writefile /tmp/src/submission.py
"""Write matching_results.tsv / candidate_pairs.tsv in the exact challenge format.

One row per S1 (in the given order), tab-separated, comma-joined IDs, no quoting,
no duplicates, empty second column when there is nothing to list.
"""
from collections.abc import Iterable, Mapping
from pathlib import Path

MATCHING_HEADER = ("source1_entity_id", "matched_entity_ids")
CANDIDATE_HEADER = ("source1_entity_id", "candidate_entity_ids")


def write_id_lists(path: Path, header: tuple[str, str], s1_ids: Iterable[str],
                   lists: Mapping[str, Iterable[str]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\t".join(header) + "\n")
        for s1 in s1_ids:
            ids = list(dict.fromkeys(i for i in lists.get(s1, ()) if not i.startswith("S1-")))
            f.write(f"{s1}\t{','.join(ids)}\n")
            n += 1
    return n


def write_submission(out_dir: Path, s1_ids: list[str], matches: Mapping[str, Iterable[str]],
                     candidates: Mapping[str, Iterable[str]]) -> None:
    for s1, m in matches.items():
        extra = set(m) - set(candidates.get(s1, ()))
        if extra:
            raise ValueError(f"{s1}: matches not in candidates: {sorted(extra)[:5]}")
    write_id_lists(out_dir / "matching_results.tsv", MATCHING_HEADER, s1_ids, matches)
    write_id_lists(out_dir / "candidate_pairs.tsv", CANDIDATE_HEADER, s1_ids, candidates)


In [ ]:
%%writefile /tmp/src/two_stage.py
"""Two-stage match model: stage-1 LightGBM (feature set C) -> set-level features -> stage-2 LightGBM.

Stage 1 gives every candidate pair a probability p1: out-of-fold on train (5 folds grouped by S1), the average
of the fold models on validation and test. Set features describe a pair relative to the other candidates of the
same S1 list (list-local, so they mean the same on train, validation and test):
  s_p1           stage-1 probability
  s_rank / s_gap rank of p1 in the S1's list, and the gap to the S1's best p1
  s_n05 / s_n09  number of candidates of the S1 with p1 > 0.5 / > 0.9
  s_sim_name / s_sim_addr / s_num_eq
                 the pair's candidate vs the S1's OTHER confident candidates (p1 > HIGH): max name token-set
                 ratio, max address token-set ratio, and whether any shares the first house number. A second
                 record of the same business looks like the first one; a decoy differs in number or legal form.
Cross-S1 counts (how many S1 lists a record is in) are deliberately NOT used: they depend on which S1 were
queried (300k of 2.2M train S1 vs all test S1), so they would shift between train and test. The reverse-search
rank / gap features already compare S1 for a record with all S1 competing.
"""
from __future__ import annotations

import lightgbm as lgb
import numpy as np
import polars as pl
from rapidfuzz import fuzz, process

SEED = 42
HIGH = 0.8
N_FOLDS = 5
SET_FEATURES = ["s_p1", "s_rank", "s_gap", "s_n05", "s_n09", "s_sim_name", "s_sim_addr", "s_num_eq"]
PARAMS = dict(objective="binary", learning_rate=0.08, num_leaves=127, min_data_in_leaf=100, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=0)
ROUNDS, EARLY_STOP = 4000, 100


def fold_of(s1: pl.Series, n: int = N_FOLDS, salt: int = 0) -> np.ndarray:
    return (s1.hash(seed=SEED + salt) % n).cast(pl.Int64).to_numpy()


def fit(m: np.ndarray, y: np.ndarray, hold: np.ndarray, feats: list[str], seed: int = SEED, params: dict | None = None) -> lgb.Booster:
    p = {**PARAMS, **(params or {}), "seed": seed}
    dtr = lgb.Dataset(m[~hold], y[~hold], feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(m[hold], y[hold], reference=dtr)
    return lgb.train(p, dtr, ROUNDS, valid_sets=[dva], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])


def predict(b: lgb.Booster, m: np.ndarray, chunk: int = 2_000_000) -> np.ndarray:
    return np.concatenate([b.predict(m[lo:lo + chunk], num_iteration=b.best_iteration) for lo in range(0, len(m), chunk)]
                          or [np.empty(0)])


def stage1_oof(x: pl.DataFrame, y: np.ndarray, feats: list[str], log=print) -> tuple[np.ndarray, list[lgb.Booster]]:
    """5-fold out-of-fold p1 on train (folds by S1); each fold model early-stops on a tenth of its own S1."""
    fold = fold_of(x["s1"])
    es = fold_of(x["s1"], 10, salt=1) == 0
    m = x.select(feats).to_numpy()
    oof, models = np.zeros(len(y)), []
    for k in range(N_FOLDS):
        tr = fold != k
        b = fit(m[tr], y[tr], es[tr], feats)
        oof[~tr] = predict(b, m[~tr])
        models.append(b)
        log(f"  stage-1 fold {k}: best iteration {b.best_iteration}")
    return oof, models


def stage1_predict(models: list[lgb.Booster], x: pl.DataFrame, feats: list[str]) -> np.ndarray:
    m = x.select(feats).to_numpy()
    return np.mean([predict(b, m) for b in models], axis=0)


def set_features(pairs: pl.DataFrame, p1: np.ndarray, rec: pl.DataFrame) -> pl.DataFrame:
    """pairs: s1, cand (all candidates of each S1 present); rec: entity_id, full_name, addr_norm, numbers (S2/S3
    records of the candidates). Returns s1, cand + SET_FEATURES, in the order of pairs."""
    x = pairs.select("s1", "cand").with_row_index("_i").with_columns(pl.Series("s_p1", p1, dtype=pl.Float64))
    x = x.with_columns(
        pl.col("s_p1").rank("ordinal", descending=True).over("s1").cast(pl.Float32).alias("s_rank"),
        (pl.col("s_p1").max().over("s1") - pl.col("s_p1")).alias("s_gap"),
        (pl.col("s_p1") > 0.5).sum().over("s1").cast(pl.Float32).alias("s_n05"),
        (pl.col("s_p1") > 0.9).sum().over("s1").cast(pl.Float32).alias("s_n09"),
    )
    r = rec.select(pl.col("entity_id").alias("cand"), pl.col("full_name").fill_null(""), pl.col("addr_norm").fill_null(""),
                   pl.col("numbers").list.first().alias("num1"))
    a = x.select("_i", "s1", "cand").join(r, on="cand", how="left")
    hi = x.filter(pl.col("s_p1") > HIGH).select("s1", pl.col("cand").alias("cand_o")).join(
        r.rename({"cand": "cand_o", "full_name": "name_o", "addr_norm": "addr_o", "num1": "num1_o"}), on="cand_o", how="left")
    pr = a.join(hi, on="s1", how="inner").filter(pl.col("cand") != pl.col("cand_o"))
    if pr.height:
        pr = pr.with_columns(
            pl.Series("sn", process.cpdist(pr["full_name"].to_list(), pr["name_o"].to_list(), scorer=fuzz.token_set_ratio, workers=-1) / 100.0),
            pl.Series("sa", process.cpdist(pr["addr_norm"].to_list(), pr["addr_o"].to_list(), scorer=fuzz.token_set_ratio, workers=-1) / 100.0),
            (pl.col("num1").is_not_null() & (pl.col("num1") == pl.col("num1_o"))).cast(pl.Float32).alias("ne"))
        agg = pr.group_by("_i").agg(pl.col("sn").max().alias("s_sim_name"), pl.col("sa").max().alias("s_sim_addr"),
                                    pl.col("ne").max().alias("s_num_eq"))
    else:
        agg = pl.DataFrame(schema={"_i": pl.UInt32, "s_sim_name": pl.Float64, "s_sim_addr": pl.Float64, "s_num_eq": pl.Float32})
    out = x.join(agg, on="_i", how="left").sort("_i")
    return out.select("s1", "cand", *[pl.col(c).cast(pl.Float32) for c in SET_FEATURES])


In [ ]:
%%writefile /tmp/src/metric.py
"""Macro F0.5 exactly as the challenge defines it.

Per S1 entity: F0.5 = 1.25*P*R / (0.25*P + R).
- No true matches (singleton): 1.0 if the prediction is empty, else 0.0.
- True matches but empty prediction, or no correct IDs: 0.0.
Macro average over ALL S1 entities in the evaluation set; an S1 missing from the
predictions counts as an empty prediction.
"""
from collections.abc import Iterable, Mapping

import polars as pl

BETA2 = 0.25  # beta = 0.5


def parse_ids(s: str | None) -> set[str]:
    """'S2-1,S3-2' -> {'S2-1', 'S3-2'}; '' or None -> set()."""
    return {x.strip() for x in s.split(",") if x.strip()} if s else set()


def f05(pred: Iterable[str], truth: Iterable[str]) -> float:
    pred, truth = set(pred), set(truth)
    if not truth:
        return 1.0 if not pred else 0.0
    tp = len(pred & truth)
    if tp == 0:
        return 0.0
    p, r = tp / len(pred), tp / len(truth)
    return (1 + BETA2) * p * r / (BETA2 * p + r)


def per_entity(pred: Mapping[str, Iterable[str]], truth: Mapping[str, Iterable[str]]) -> pl.DataFrame:
    """One row per S1 in `truth`: f05, precision, recall (P/R null when undefined)."""
    rows = []
    for s1, t in truth.items():
        p, t = set(pred.get(s1, ())), set(t)
        tp = len(p & t)
        rows.append({
            "s1": s1,
            "f05": f05(p, t),
            "precision": tp / len(p) if p else None,
            "recall": tp / len(t) if t else None,
            "n_true": len(t),
            "n_pred": len(p),
        })
    return pl.DataFrame(rows, schema={"s1": pl.String, "f05": pl.Float64, "precision": pl.Float64,
                                      "recall": pl.Float64, "n_true": pl.Int64, "n_pred": pl.Int64})


def report(pred: Mapping[str, Iterable[str]], truth: Mapping[str, Iterable[str]],
           country: Mapping[str, str] | None = None) -> pl.DataFrame:
    """Macro F0.5 overall, per country, and for singletons vs non-singletons."""
    df = per_entity(pred, truth).with_columns(
        pl.when(pl.col("n_true") == 0).then(pl.lit("singleton")).otherwise(pl.lit("non_singleton")).alias("kind"),
        pl.col("s1").replace_strict(country or {}, default="ALL").alias("country"),
    )
    aggs = [
        pl.len().alias("s1"),
        pl.col("f05").mean().alias("macro_f05"),
        pl.col("precision").mean().alias("mean_precision"),
        pl.col("recall").mean().alias("mean_recall"),
    ]
    parts = [df.select(pl.lit("overall").alias("group"), *aggs)]
    if country:
        parts.append(df.group_by("country").agg(aggs).sort("country").select(pl.format("country={}", "country").alias("group"), *[a.meta.output_name() for a in aggs]))
    parts.append(df.group_by("kind").agg(aggs).sort("kind").select(pl.format("kind={}", "kind").alias("group"), *[a.meta.output_name() for a in aggs]))
    return pl.concat(parts, how="vertical_relaxed")


In [ ]:
%%writefile /tmp/src/validate_submission.py
#!/usr/bin/env python3
"""
ML Challenge 2026 — Submission Validator

Run this BEFORE submitting. It checks your output files against every formatting
rule the scorer enforces, so you can catch a rejection locally instead of burning
a submission. It reads only your output files and the test source files (to learn
which S1 entities are required and which S2/S3 IDs exist); it never needs the
ground truth and never computes your score.

It validates two files:

* ``matching_results.tsv`` (required) — your final matches, the file scored on the
  leaderboard.
* ``candidate_pairs.tsv`` (optional) — the candidate set from your blocking stage.
  When present, the validator also checks that your final matches are a subset of
  your candidates and *warns* (never fails) otherwise. When absent it is skipped
  with a warning; it is still expected in your final submission zip.

Stdlib only, Python 3.8+. Run from the ``student_resource/`` directory::

    python3 utils/validate_submission.py \
        --matching output/matching_results.tsv \
        --candidate output/candidate_pairs.tsv \
        --test-dir dataset/test

Exit code 0 means the files are safe to submit; 1 means fix the listed issues
(warnings never fail the run).

ID-existence check (off by default). By default the validator does NOT check that
every matched/candidate ID actually exists in the test set: that check loads all
Source-2/3 IDs into memory, which on the full ~1.7M-entity test set costs a few GB
(more when ``candidate_pairs.tsv`` is included). The default run therefore stays fast
and light and verifies every other rule; it prints a warning noting the check was
skipped. Pass ``--check-ids`` to turn it on (it reads ``test_source2.tsv`` /
``test_source3.tsv`` from ``--test-dir``); a missing/garbage matched ID only lowers
your score rather than being rejected by the scorer, so this check is a diagnostic,
not a gate. If ``--check-ids`` runs out of memory, drop ``--candidate`` (the candidate
cross-check is the biggest memory user, and the matching file is the only one scored).
"""

import argparse
import os
import sys

DELIM = "\t"
MAX_EXAMPLES = 5  # how many offending IDs to show per issue
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    """Return the set of first-column entity IDs from a source TSV.

    The header row is skipped and blank lines are ignored.
    """
    with open(path, encoding="utf-8") as f:
        next(f, None)  # skip header
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    """Return a short, human-readable sample of ``items`` for an error message."""
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def load_match_targets(test_dir, warnings):
    """Return the set of valid S2/S3 match IDs, or ``None`` if unavailable.

    Only called when ``--check-ids`` is on. When ``test_source2.tsv`` or
    ``test_source3.tsv`` is missing we cannot check that matched IDs exist, so we
    record a warning and return ``None`` to signal that the existence check should be
    skipped.
    """
    targets = set()
    for name in ("test_source2.tsv", "test_source3.tsv"):
        path = os.path.join(test_dir, name)
        if not os.path.isfile(path):
            warnings.append(
                f"{path} not found — skipping the (optional) check that matched "
                f"IDs exist in the test set. Every other rule is still checked. "
                f"This is the lighter-memory mode; provide test_source2/3.tsv to "
                f"enable the ID-existence check."
            )
            return None
        targets |= read_ids(path)
    return targets


def validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):
    """Validate one results-style TSV (matching or candidate).

    Applies the shared formatting rules and appends any problems to ``errors``.
    Returns a ``{source1_id: set(matched/candidate ids)}`` mapping, or ``None`` on a
    fatal problem (missing file, empty file, or a broken header) that stops parsing.
    """
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None
        if DELIM not in header and "," in header:  # the #1 mistake: a CSV
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv); "
                "write it with df.to_csv(sep='\\t', index=False)."
            )
            return None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None

        for line_num, line in enumerate(f, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            mapping[s1] = id_set
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    # Aggregate the per-category findings. Each entry is (offenders, message);
    # only non-empty categories become errors.
    findings = [
        (
            dup_rows,
            "{name}: duplicate source1_entity_id row(s): {ex}. "
            "Each S1 entity may appear on only one row.",
        ),
        (
            intra_dupes,
            "{name}: repeated ID inside a {col} list for: {ex}. "
            "No duplicate IDs are allowed within a list.",
        ),
        (
            self_matches,
            "{name}: {col} contains Source-1 IDs (self-matches): {ex}. "
            "Only S2-/S3- IDs are allowed.",
        ),
        (
            wrong_prefix,
            "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}.",
        ),
        (
            unknown,
            "{name}: {col} references IDs not in the test "
            "Source-2/3 files: {ex}.",
        ),
        (
            required - seen,
            "{name}: required S1 entity(ies) missing: {ex}. "
            "Every entity in test_source1.tsv needs a row (empty = no match).",
        ),
        (
            seen - required,
            "{name}: row(s) using an S1 ID that is not in the test set: {ex}.",
        ),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping


def validate(matching_path, candidate_path, test_dir, check_ids=False):
    """Validate the submission output(s); return ``(errors, warnings)`` lists.

    ``check_ids`` (``--check-ids``) turns on the optional, memory-heavy check that
    every matched/candidate ID exists in the test Source-2/3 files. It is off by
    default so the common run stays fast and light.
    """
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    if check_ids:
        valid_ids = load_match_targets(test_dir, warnings)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        valid_ids = None
        warnings.append(
            "ID-existence check is OFF (the default) — not checking that matched/"
            "candidate IDs exist in the test set. Every other rule is still checked. "
            "Re-run with --check-ids to enable it (needs test_source2/3.tsv; uses "
            "more memory). A nonexistent ID only lowers your score, never rejects "
            "your submission."
        )

    matched = validate_id_list_file(
        matching_path, MATCHING_HEADER, "matched_entity_ids", required, valid_ids, errors
    )

    # candidate_pairs.tsv is optional: if it's absent we skip its checks with a
    # warning (it's still expected in your final submission zip). A missing
    # candidate file never fails this run on its own.
    candidate = None
    if candidate_path and os.path.isfile(candidate_path):
        candidate = validate_id_list_file(
            candidate_path, CANDIDATE_HEADER, "candidate_entity_ids",
            required, valid_ids, errors,
        )
    elif candidate_path:
        warnings.append(
            f"{candidate_path} not found — skipping candidate_pairs.tsv checks. "
            "It is optional here, but your final submission zip must include "
            "output/candidate_pairs.tsv."
        )

    # Soft check: your final matches should come from your blocking candidates.
    # A matched ID absent from candidate_pairs.tsv usually means a pipeline bug,
    # so we warn but never fail on it.
    if matched is not None and candidate is not None:
        offenders = {
            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())
        }
        if offenders:
            warnings.append(
                f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                f"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches "
                "normally come from your blocking candidates — double-check these."
            )

    return errors, warnings


def main():
    parser = argparse.ArgumentParser(
        description="Validate ML Challenge 2026 submission output files before submitting."
    )
    parser.add_argument(
        "--matching",
        "-m",
        default="output/matching_results.tsv",
        help="Path to matching_results.tsv (default: %(default)s)",
    )
    parser.add_argument(
        "--candidate",
        "-c",
        default=None,
        help="Path to candidate_pairs.tsv "
        "(default: output/candidate_pairs.tsv if it exists).",
    )
    parser.add_argument(
        "--test-dir",
        "-t",
        default="dataset/test",
        help="Folder with test_source1/2/3.tsv (default: %(default)s). "
        "test_source2/3.tsv are only read when --check-ids is given.",
    )
    parser.add_argument(
        "--check-ids",
        action="store_true",
        help="Also check that every matched/candidate ID exists in the test "
        "Source-2/3 files. Off by default (loads all S2/S3 IDs into memory — a few "
        "GB on the full test set). A nonexistent ID only lowers your score, so this "
        "is a diagnostic, not a submission gate.",
    )
    args = parser.parse_args()

    # candidate_pairs.tsv is optional; default to the conventional path and let
    # validate() skip (with a warning) if the file isn't there.
    candidate_path = args.candidate or "output/candidate_pairs.tsv"

    print("ML Challenge 2026 — submission validator")
    print(f"  test dir: {args.test_dir}")
    try:
        errors, warnings = validate(
            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids
        )
    except UnicodeDecodeError:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(
            f"  1. A file is not valid UTF-8 text (most likely {args.matching} or "
            f"{candidate_path}). Re-save it as a plain UTF-8, tab-separated .tsv — "
            "not cp1252/Latin-1, and not a compressed or binary file (.gz/.xlsx/"
            ".parquet) renamed to .tsv. In pandas: "
            "df.to_csv(path, sep='\\t', index=False, encoding='utf-8')."
        )
        return 1
    except OSError as exc:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(f"  1. Could not read a file: {exc}.")
        return 1

    print()
    for warning in warnings:
        print(f"WARNING: {warning}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        return 1
    print("PASS — no blocking issues found. Safe to submit.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import sys
sys.path.insert(0, '/tmp/src')
sys.argv = ['run_two_stage.py', '--input', '/kaggle/input', '--out-dir', '/kaggle/working', '--validator', '/tmp/src/validate_submission.py', '--seeds', '3']

In [ ]:
"""Model v2 experiments (blocking, candidates and features-v2 unchanged): error analysis, selection rules,
two-stage model with set features, seed averaging; then test outputs with the best setup.

Validation protocol as in run_model.py: selection tuned on the tune half (even S1 id), reported on the report
half (odd id); every validation S1 counts. A setup is kept only if report-half F0.5 beats the v1 reference
overall AND for US and India. Training S1 exclude the pruner's 100k (in-sample pruner features).

Usage: python src/run_two_stage.py --input DIR --out-dir DIR --validator utils/validate_submission.py [--seeds 3]
"""
import argparse
import json
import resource
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import polars as pl

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:  # Kaggle notebook: modules are written to /tmp/src
    sys.path.insert(0, "/tmp/src")
from model import feature_sets  # noqa: E402
from selection import f05_table, select, summary  # noqa: E402
from submission import write_submission  # noqa: E402
from two_stage import SET_FEATURES, fit, fold_of, predict, set_features, stage1_oof, stage1_predict  # noqa: E402

SEED = 42
REF = {"overall": 0.9717121937196413, "country=US": 0.9804, "country=India": 0.9586}  # model-v1 variant C, report half
TAUS = [round(x, 3) for x in np.arange(0.30, 0.96, 0.025)]
T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.time() - T0:7.1f}s peak {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:5.1f} GB] {msg}", flush=True)


def find(root: Path, name: str, parent: str | None = None) -> Path:
    hits = sorted(p for p in root.rglob(name) if parent is None or p.parent.name == parent)
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]


def md(df: pl.DataFrame, n: int = 60) -> str:
    df = df.head(n)
    f = lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)
    rows = [" | ".join(df.columns), " | ".join("---" for _ in df.columns)] + [" | ".join(f(v) for v in r) for r in df.iter_rows()]
    return "\n".join(f"| {r} |" for r in rows)


def tune_all(sv: pl.DataFrame, truth, s1s, rules: bool) -> pl.DataFrame:
    """Grid over tau, margin, one_owner (+ tau2 secondary rule when set features exist)."""
    rows = []
    for tau in TAUS:
        for margin in (0.0, 0.5, 0.7, 0.8):
            for owner in (False, True):
                for tau2 in ([None] + ([round(tau - d, 3) for d in (0.1, 0.2, 0.3) if tau - d > 0.05] if rules else [])):
                    cfg = dict(tau=tau, margin=margin, one_owner=owner, tau2=tau2)
                    t = f05_table(select(sv, **cfg), truth, s1s)
                    rows.append({**cfg, "macro_f05": t["f05"].mean()})
    return pl.DataFrame(rows).sort("macro_f05", descending=True)


def evaluate(name: str, sv: pl.DataFrame, truth, s1_tune, s1_rep, rules: bool) -> dict:
    grid = tune_all(sv, truth, s1_tune, rules)
    best = {k: grid.row(0, named=True)[k] for k in ("tau", "margin", "one_owner", "tau2")}
    sm = summary(f05_table(select(sv, **best), truth, s1_rep), by="country")
    g = {r["group"]: r for r in sm.iter_rows(named=True)}
    res = {"setup": name, **best, "tune_f05": grid["macro_f05"][0],
           **{k: g[k]["macro_f05"] for k in g}, "precision": g["overall"]["precision"], "recall": g["overall"]["recall"]}
    res["kept_vs_v1"] = all(res.get(k, 0) > v for k, v in REF.items())
    log(f"{name}: {best} -> report F0.5 {res['overall']:.4f} (US {res['country=US']:.4f}, India {res['country=India']:.4f})")
    return res


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--out-dir", required=True, type=Path)
    ap.add_argument("--validator", default=None)
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--skip-test", action="store_true")
    ap.add_argument("--pruner-s1", type=int, default=100_000)
    a = ap.parse_args()
    out = a.out_dir
    out.mkdir(parents=True, exist_ok=True)
    rd = lambda name, cols=None, parent=None: pl.read_parquet(find(a.input, name, parent), columns=cols)
    raw = lambda name: sorted(p for p in a.input.rglob(name) if p.parent.name != "normalised")[0]  # organiser copy, not norm-v3

    # ------------------------------------------------------------------ data
    split = rd("g25_split.parquet")
    trn = split.filter(pl.col("role") == "train")["source1_entity_id"]
    pruner_s1 = trn.sample(min(a.pruner_s1, trn.len()), seed=SEED)
    val_s1 = split.filter(pl.col("role") == "val").select(pl.col("source1_entity_id").alias("s1")).join(
        pl.read_parquet(raw("train_source1.parquet"), columns=["entity_id", "country"]).rename({"entity_id": "s1"}), on="s1", how="left")
    gt = rd("train_ground_truth.parquet")
    truth = (gt.join(val_s1, left_on="source1_entity_id", right_on="s1", how="semi")
             .select(pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").str.split(",").alias("cand"))
             .explode("cand", empty_as_null=True).filter(pl.col("cand").is_not_null() & (pl.col("cand") != "")))
    half = pl.col("s1").str.extract(r"(\d+)$").cast(pl.Int64) % 2
    s1_tune, s1_rep = val_s1.filter(half == 0), val_s1.filter(half == 1)

    ftr = rd("features_train.parquet")
    feats = feature_sets(ftr.columns)["C"]
    ftr = ftr.join(rd("train_candidates.parquet", ["s1", "cand", "is_match"]), on=["s1", "cand"], how="left")
    ftr = ftr.filter(~pl.col("s1").is_in(pruner_s1.implode())).sort("s1", "cand")
    y = ftr["is_match"].cast(pl.Int8).to_numpy()
    fva = rd("features_val.parquet").sort("s1", "cand")
    fva = fva.join(truth.with_columns(pl.lit(True).alias("is_match")), on=["s1", "cand"], how="left").with_columns(pl.col("is_match").fill_null(False))
    log(f"train {ftr.height:,} pairs / {ftr['s1'].n_unique():,} S1; val {fva.height:,} pairs; {len(feats)} features (set C)")

    # candidate records (S2/S3) for set features and examples
    cols = ["entity_id", "business_name", "business_address", "full_name", "addr_norm", "numbers"]
    ids = pl.concat([ftr["cand"], fva["cand"]]).unique().implode()
    rec_tr = pl.concat([pl.scan_parquet(find(a.input, f"train_source{k}.parquet", "normalised")).select(cols)
                        .filter(pl.col("entity_id").is_in(ids)).collect() for k in (2, 3)])
    s1_rec = pl.scan_parquet(find(a.input, "train_source1.parquet", "normalised")).select("entity_id", "business_name", "business_address") \
        .filter(pl.col("entity_id").is_in(val_s1["s1"].implode())).collect()

    # ------------------------------------------------------------------ stage 1 (OOF on train)
    oof, models1 = stage1_oof(ftr.select("s1", *feats), y, feats, log=log)
    p1_val = stage1_predict(models1, fva, feats)
    sv1 = fva.select("s1", "cand", "is_match").with_columns(pl.Series("p", p1_val))
    results = [evaluate("stage1 (C, 5-fold avg)", sv1, truth, s1_tune, s1_rep, rules=False)]

    # ------------------------------------------------------------------ step 1: error analysis (report half)
    cfg1 = {k: results[0][k] for k in ("tau", "margin", "one_owner", "tau2")}
    sel = select(sv1, **cfg1).select("s1", "cand", pl.lit(True).alias("sel"))
    ea = (sv1.join(s1_rep, on="s1", how="semi").join(sel, on=["s1", "cand"], how="left").with_columns(pl.col("sel").fill_null(False))
          .join(val_s1, on="s1").join(fva.select("s1", "cand", "f_num_jacc", "f_addr_tsr", "f_name_full_tsort", "f_name_nolegal_ratio",
                                                  "f_legal_same", "f_addr_missing", "f_num_only_s1", "f_num_only_cand"), on=["s1", "cand"]))
    fn = ea.filter(pl.col("is_match") & ~pl.col("sel"))
    fp = ea.filter(~pl.col("is_match") & pl.col("sel"))
    band = (pl.when(pl.col("p") < 0.1).then(pl.lit("<0.1")).when(pl.col("p") < 0.3).then(pl.lit("0.1-0.3"))
            .when(pl.col("p") < 0.5).then(pl.lit("0.3-0.5")).when(pl.col("p") < cfg1["tau"]).then(pl.lit(f"0.5-{cfg1['tau']}"))
            .otherwise(pl.lit(">=tau (lost to margin / owner)")))
    fn_tab = fn.with_columns(band.alias("band"), pl.col("cand").str.slice(0, 2).alias("source")).group_by("country", "source", "band").len().sort("country", "source", "band")
    names = lambda d: (d.join(s1_rec.rename({"entity_id": "s1", "business_name": "s1_name", "business_address": "s1_addr"}), on="s1", how="left")
                       .join(rec_tr.select(pl.col("entity_id").alias("cand"), pl.col("business_name").alias("c_name"),
                                           pl.col("business_address").alias("c_addr")), on="cand", how="left"))
    fn_ex = names(fn.sort("p", descending=True).sample(min(30, fn.height), seed=SEED)).select(
        "country", "p", "s1_name", "c_name", "s1_addr", "c_addr", "f_num_jacc", "f_addr_tsr", "f_name_full_tsort", "f_legal_same")
    fp_ex = names(fp.sort("p", descending=True).head(30)).select(
        "country", "p", "s1_name", "c_name", "s1_addr", "c_addr", "f_num_jacc", "f_addr_tsr", "f_name_full_tsort", "f_legal_same")
    fn_tab.write_csv(out / "ea_missed_by_band.csv"); fn_ex.write_csv(out / "ea_missed_examples.csv"); fp_ex.write_csv(out / "ea_false_pos_examples.csv")
    ea_summary = {"missed_true_candidates": fn.height, "false_positives": fp.height, "report_true_pairs": int(truth.join(s1_rep, on="s1", how="semi").height),
                  "missed_by_country": dict(fn.group_by("country").len().iter_rows()), "fp_by_country": dict(fp.group_by("country").len().iter_rows())}
    log(f"error analysis: {ea_summary}")

    # ------------------------------------------------------------------ step 3: set features + stage 2
    sf_tr = set_features(ftr, oof, rec_tr)
    sf_va = set_features(fva, p1_val, rec_tr)
    x2 = ftr.select("s1", *feats).hstack(sf_tr.select(SET_FEATURES))
    f2 = feats + SET_FEATURES
    m2 = x2.select(f2).to_numpy()
    es = fold_of(ftr["s1"], 10, salt=7) == 0
    mv = fva.select(feats).hstack(sf_va.select(SET_FEATURES)).select(f2).to_numpy()
    models2, pv = [], []
    for s in range(a.seeds):
        b = fit(m2, y, es, f2, seed=SEED + s)
        models2.append(b)
        pv.append(predict(b, mv))
        log(f"  stage-2 seed {s}: best iteration {b.best_iteration}")
        if s == 0:
            sv2a = fva.select("s1", "cand", "is_match").hstack(sf_va.select("s_sim_name", "s_num_eq")).with_columns(pl.Series("p", pv[0]))
            results.append(evaluate("stage2 (1 seed)", sv2a, truth, s1_tune, s1_rep, rules=True))
    sv2 = fva.select("s1", "cand", "is_match").hstack(sf_va.select("s_sim_name", "s_num_eq")).with_columns(pl.Series("p", np.mean(pv, axis=0)))
    results.append(evaluate(f"stage2 ({a.seeds} seeds avg)", sv2, truth, s1_tune, s1_rep, rules=True))
    res = pl.DataFrame(results, infer_schema_length=None)
    res.write_csv(out / "results.csv")
    imp = (pl.DataFrame({"feature": f2, "gain": models2[0].feature_importance("gain").astype(float)})
           .with_columns((pl.col("gain") / pl.col("gain").sum()).alias("share")).sort("gain", descending=True))
    imp.write_csv(out / "importance_stage2.csv")

    # calibration (tune half) of the best stage-2 probabilities
    cal = (sv2.join(s1_tune, on="s1", how="semi").with_columns((pl.col("p") * 10).floor().clip(0, 9).alias("bin"))
           .group_by("bin").agg(pl.len().alias("pairs"), pl.col("p").mean().alias("mean_p"), pl.col("is_match").mean().alias("match_rate")).sort("bin"))
    cal.write_csv(out / "calibration.csv")

    best_row = res.sort("overall", descending=True).row(0, named=True)
    chosen = best_row["setup"]
    cfg = {k: best_row[k] for k in ("tau", "margin", "tau2")}
    report = ["# Model v2: two-stage + selection rules\n", md(res) + "\n",
              f"Chosen: **{chosen}** {cfg} (one owner is always applied on test: an S2/S3 matches at most one S1, and on "
              "test all S1 compete, unlike validation).\n",
              "## Error analysis (stage 1, report half)\n", "```\n" + json.dumps(ea_summary, indent=1) + "\n```\n",
              md(fn_tab, 80) + "\n", "### Missed true pairs (30 random)\n", md(fn_ex, 30) + "\n",
              "### Highest-probability false positives\n", md(fp_ex, 30) + "\n",
              "## Stage-2 importance (top 25)\n", md(imp, 25) + "\n", "## Calibration (tune half)\n", md(cal) + "\n"]

    # ------------------------------------------------------------------ test
    if not a.skip_test:
        fte = rd("features_test.parquet").sort("s1", "cand")
        tids = fte["cand"].unique().implode()
        rec_te = pl.concat([pl.scan_parquet(find(a.input, f"test_source{k}.parquet", "normalised")).select(cols)
                            .filter(pl.col("entity_id").is_in(tids)).collect() for k in (2, 3)])
        p1_te = stage1_predict(models1, fte, feats)
        if chosen.startswith("stage1"):
            st = fte.select("s1", "cand").with_columns(pl.Series("p", p1_te), pl.lit(None, pl.Float32).alias("s_sim_name"), pl.lit(None, pl.Float32).alias("s_num_eq"))
        else:
            sf_te = set_features(fte, p1_te, rec_te)
            mt = fte.select(feats).hstack(sf_te.select(SET_FEATURES)).select(f2).to_numpy()
            nm = 1 if "1 seed" in chosen else len(models2)
            st = fte.select("s1", "cand").hstack(sf_te.select("s_sim_name", "s_num_eq")).with_columns(
                pl.Series("p", np.mean([predict(b, mt) for b in models2[:nm]], axis=0)))
        matched = select(st, tau=cfg["tau"], margin=cfg["margin"], one_owner=True, tau2=cfg["tau2"])
        s1_all = pl.read_parquet(raw("test_source1.parquet"), columns=["entity_id", "country"]).rename({"entity_id": "s1"})
        cand = rd("test_candidates.parquet", ["s1", "cand", "p_u50"])
        lists = cand.sort(["s1", "p_u50", "cand"], descending=[False, True, False]).group_by("s1", maintain_order=True).agg(pl.col("cand"))
        ml = matched.sort(["s1", "p"], descending=[False, True]).group_by("s1", maintain_order=True).agg(pl.col("cand"))
        write_submission(out / "output", s1_all["s1"].to_list(), dict(zip(ml["s1"].to_list(), ml["cand"].to_list())),
                         dict(zip(lists["s1"].to_list(), lists["cand"].to_list())))
        vm = None
        if a.validator:
            import importlib.util
            spec = importlib.util.spec_from_file_location("vs", a.validator); vs = importlib.util.module_from_spec(spec); spec.loader.exec_module(vs)
            with tempfile.TemporaryDirectory() as td:
                for k in (1, 2, 3):
                    pl.read_parquet(raw(f"test_source{k}.parquet"), columns=["entity_id"]).write_csv(Path(td) / f"test_source{k}.tsv", separator="\t")
                e, w = vs.validate(str(out / "output/matching_results.tsv"), str(out / "output/candidate_pairs.tsv"), td, check_ids=True)
            vm = {"errors": e, "warnings": w}
            log(f"validator: {len(e)} errors, {len(w)} warnings")
        n = s1_all.join(matched.group_by("s1").len("n"), on="s1", how="left").with_columns(pl.col("n").fill_null(0))
        pc = n.group_by("country").agg(pl.col("n").mean().alias("matches_per_s1"), (pl.col("n") == 0).mean().alias("share_no_match")).sort("country")
        fr = s1_all.filter(pl.col("country") == "France").sample(10, seed=SEED)
        s1te = pl.scan_parquet(find(a.input, "test_source1.parquet", "normalised")).select("entity_id", "business_name", "business_address") \
            .filter(pl.col("entity_id").is_in(fr["s1"].implode())).collect()
        ex = (matched.join(fr, on="s1", how="semi").join(rec_te.select(pl.col("entity_id").alias("cand"), pl.col("business_name").alias("c_name"),
                                                                        pl.col("business_address").alias("c_addr")), on="cand", how="left")
              .join(s1te.rename({"entity_id": "s1"}), on="s1", how="right").select("s1", "business_name", "business_address", "p", "c_name", "c_addr"))
        pc.write_csv(out / "test_matches_per_country.csv"); ex.write_csv(out / "test_france_examples.csv")
        report += ["## Test\n", f"Matches {matched.height:,}; validator {vm}.\n", md(pc) + "\n", "### 10 random French S1\n", md(ex, 60) + "\n"]
        (out / "test_check.json").write_text(json.dumps({"validator": vm, "matches": matched.height}, default=str))
    (out / "report.md").write_text("\n".join(report))
    (out / "config.json").write_text(json.dumps({"chosen": chosen, **cfg, "one_owner_test": True, "features": f2,
                                                 "results": results}, indent=1, default=str))
    for i, b in enumerate(models1):
        b.save_model(str(out / f"stage1_fold{i}.txt"), num_iteration=b.best_iteration)
    for i, b in enumerate(models2):
        b.save_model(str(out / f"stage2_seed{i}.txt"), num_iteration=b.best_iteration)
    sv2.join(sv1.select("s1", "cand", pl.col("p").alias("p1")), on=["s1", "cand"]).write_parquet(out / "val_scores.parquet")
    log("done")


if __name__ == "__main__":
    main()
